In [10]:
!pip install rdkit

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 71.2 MB/s eta 0:00:0000:0100:01


In [1]:
import h5py, pickle, numpy as np
import scanpy as sc
import anndata as ad
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator as rfpgen
import numpy as np, pickle, sys
from tqdm.auto import tqdm

In [2]:
with open('../data/LINCS2020/data_example/ECFP4_emb2048.pickle', 'rb') as f:
    smi2emb = pickle.load(f)

In [3]:
smi2emb

{'BrC1C(Br)C(Br)C(Br)C(Br)C1Br': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'BrCC(=O)NCCc1c[nH]c2ccccc12': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'BrCC(=O)NCCc1ccc2ccccc2c1': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'BrCC(=O)NCCc1ccccc1': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'Brc1c(Br)c(Br)c2[nH]nnc2c1Br': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'Brc1c(NC2=NCCN2)ccc2nccnc12': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'Brc1cc2OCOc2cc1C3Nc4ccccc4C5C=CCC53': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'Brc1ccc(C=CCNCCNS(=O)(=O)c2cccc3cnccc23)cc1': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'Brc1ccc(CSc2nnc(c3ccccn3)n2Cc4ccco4)cc1': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'Brc1ccc(Cn2cncc2Cn2cc(C(=O)N3CCOCC3)c(c2)-c2cccc3ccccc23)cc1': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)

In [4]:
type(smi2emb)

dict

In [8]:
smi2emb['BrC1C(Br)C(Br)C(Br)C(Br)C1Br'].dtype

dtype('float32')

In [12]:
idx2smi_path = "./Code/other_models/TranSiGen/data/Meisheng_used_data/idx2smi.pickle"

with open(idx2smi_path, "rb") as fh:
    idx2smi = pickle.load(fh)          # {int: str}

In [15]:
list(idx2smi.items())[:5]

[(0, 'CCC1(CCC(=O)NC1=O)c1ccc(N)cc1'),
 (1, 'OC(=O)CCc1nc(c(o1)-c1ccccc1)-c1ccccc1'),
 (2, 'CNC(=O)Oc1ccc2N(C)[C@H]3N(C)CC[C@@]3(C)c2c1'),
 (3, 'CC1(CN2CCC(CC2)n2c3ccccc3[nH]c2=O)OCc2ccccc2-n2cccc12'),
 (4, 'CC(C)C[C@@H](NC(=O)[C@@H](Cc1ccccc1)NC(=O)C1=CNC=CN1)B(O)O')]

In [16]:
idx2smi[0]

'CCC1(CCC(=O)NC1=O)c1ccc(N)cc1'

In [19]:
# ensure they’re in integer order 0…N-1
smiles_list = [idx2smi[i] for i in sorted(idx2smi)]
print("Unique SMILES:", len(smiles_list))
print("First 5 →")
smiles_list[:5]

Unique SMILES: 17766
First 5 →


['CCC1(CCC(=O)NC1=O)c1ccc(N)cc1',
 'OC(=O)CCc1nc(c(o1)-c1ccccc1)-c1ccccc1',
 'CNC(=O)Oc1ccc2N(C)[C@H]3N(C)CC[C@@]3(C)c2c1',
 'CC1(CN2CCC(CC2)n2c3ccccc3[nH]c2=O)OCc2ccccc2-n2cccc12',
 'CC(C)C[C@@H](NC(=O)[C@@H](Cc1ccccc1)NC(=O)C1=CNC=CN1)B(O)O']

In [20]:
# ── hyper-parameters ───────────────────────────────────────────────
RADIUS = 2          # ECFP4 → radius 2
N_BITS = 2048       # 2 048-bit output

FP_GEN = rfpgen.GetMorganGenerator(
    radius=RADIUS,
    fpSize=N_BITS,
    includeChirality=False,
    countSimulation=False      # binary bits
)

In [21]:
def morgan_bits_to_np(smiles: str):
    """Return np.float32[2048] (0/1 bits) or None if parsing fails."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    bv = FP_GEN.GetFingerprint(mol)               # ExplicitBitVect
    arr = np.zeros((N_BITS,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(bv, arr)      # fills in-place (uint8 0/1)
    return arr

In [22]:
smi2emb = {}
n_fail  = 0

for s in tqdm(smiles_list, desc="Generating ECFP4-2048"):
    arr = morgan_bits_to_np(s)
    if arr is None:
        n_fail += 1
        continue                                 # skip or handle as you wish
    smi2emb[s] = arr

print(f"Finished. Success: {len(smi2emb):,}  Failures: {n_fail}")

Generating ECFP4-2048:   0%|          | 0/17766 [00:00<?, ?it/s]

Finished. Success: 17,766  Failures: 0


In [23]:
out_path = "./Code/other_models/TranSiGen/data/Meisheng_used_data/ECFP4_emb2048.pickle"

with open(out_path, "wb") as fh:
    pickle.dump(smi2emb, fh, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved to", out_path)

Saved to /work/users/m/e/meisheng/Dissertation/TranSiGen/data/Meisheng_used_data/ECFP4_emb2048.pickle


In [ ]:
'''
From now on, all are sanity checks.
'''

In [5]:
out_path = "./Code/other_models/TranSiGen/data/Meisheng_used_data/ECFP4_emb2048.pickle"

In [6]:
with open(out_path, "rb") as fh:
    smi2emb_check = pickle.load(fh)

first_key = next(iter(smi2emb_check))
print("First key:", first_key)
print("Fingerprint dtype / shape:", smi2emb_check[first_key].dtype,
      smi2emb_check[first_key].shape)
print("Bit count (should be ≤2048):", int(smi2emb_check[first_key].sum()))

First key: CCC1(CCC(=O)NC1=O)c1ccc(N)cc1
Fingerprint dtype / shape: float32 (2048,)
Bit count (should be ≤2048): 32


In [7]:
list(smi2emb_check.items())[:5]

[('CCC1(CCC(=O)NC1=O)c1ccc(N)cc1',
  array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)),
 ('OC(=O)CCc1nc(c(o1)-c1ccccc1)-c1ccccc1',
  array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)),
 ('CNC(=O)Oc1ccc2N(C)[C@H]3N(C)CC[C@@]3(C)c2c1',
  array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)),
 ('CC1(CN2CCC(CC2)n2c3ccccc3[nH]c2=O)OCc2ccccc2-n2cccc12',
  array([0., 0., 0., ..., 0., 1., 0.], dtype=float32)),
 ('CC(C)C[C@@H](NC(=O)[C@@H](Cc1ccccc1)NC(=O)C1=CNC=CN1)B(O)O',
  array([0., 1., 0., ..., 0., 0., 0.], dtype=float32))]

In [8]:
list(smi2emb.items())[:5]

[('BrC1C(Br)C(Br)C(Br)C(Br)C1Br',
  array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)),
 ('BrCC(=O)NCCc1c[nH]c2ccccc12',
  array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)),
 ('BrCC(=O)NCCc1ccc2ccccc2c1',
  array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)),
 ('BrCC(=O)NCCc1ccccc1', array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)),
 ('Brc1c(Br)c(Br)c2[nH]nnc2c1Br',
  array([0., 0., 0., ..., 0., 0., 0.], dtype=float32))]

In [29]:
len(smi2emb_check)

17766

In [11]:
import pandas as pd

In [13]:
# ── convert to DataFrame: rows = SMILES, columns = bit_0 … bit_2047 ────
df_fp = pd.DataFrame.from_dict(smi2emb_check, orient="index")
df_fp.columns = [f"bit_{i}" for i in range(df_fp.shape[1])]
df_fp.index.name = "canonical_smiles"

# ── optional: wider display in Jupyter ──────────────────────────────────
pd.set_option("display.max_columns", 50)   # show first 20 bits; change as you like
pd.set_option("display.width", 0)          # let Jupyter decide line breaks

# ── preview ─────────────────────────────────────────────────────────────
display(df_fp.head())          # first 5 SMILES; or .head(50) for more

,bit_0,bit_1,bit_2,bit_3,bit_4,bit_5,bit_6,bit_7,bit_8,bit_9,bit_10,bit_11,bit_12,bit_13,bit_14,bit_15,bit_16,bit_17,bit_18,bit_19,bit_20,bit_21,bit_22,bit_23,bit_24,...,bit_2023,bit_2024,bit_2025,bit_2026,bit_2027,bit_2028,bit_2029,bit_2030,bit_2031,bit_2032,bit_2033,bit_2034,bit_2035,bit_2036,bit_2037,bit_2038,bit_2039,bit_2040,bit_2041,bit_2042,bit_2043,bit_2044,bit_2045,bit_2046,bit_2047
canonical_smiles,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
CCC1(CCC(=O)NC1=O)c1ccc(N)cc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
OC(=O)CCc1nc(c(o1)-c1ccccc1)-c1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CNC(=O)Oc1ccc2N(C)[C@H]3N(C)CC[C@@]3(C)c2c1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CC1(CN2CCC(CC2)n2c3ccccc3[nH]c2=O)OCc2ccccc2-n2cccc12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
CC(C)C[C@@H](NC(=O)[C@@H](Cc1ccccc1)NC(=O)C1=CNC=CN1)B(O)O,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
row_bit_sum = df_fp.sum(axis=1)

In [15]:
row_bit_sum

canonical_smiles
CCC1(CCC(=O)NC1=O)c1ccc(N)cc1                                                        32.0
OC(=O)CCc1nc(c(o1)-c1ccccc1)-c1ccccc1                                                32.0
CNC(=O)Oc1ccc2N(C)[C@H]3N(C)CC[C@@]3(C)c2c1                                          44.0
CC1(CN2CCC(CC2)n2c3ccccc3[nH]c2=O)OCc2ccccc2-n2cccc12                                58.0
CC(C)C[C@@H](NC(=O)[C@@H](Cc1ccccc1)NC(=O)C1=CNC=CN1)B(O)O                           50.0
                                                                                     ... 
CCCC(=O)Nc1ccc2OC[C@H](C)N(Cc3ccc(cc3)-c3ccccn3)C[C@H](C)[C@H](CN(C)C(=O)c2c1)OC     76.0
Cc1cc(CS(=O)(=O)c2ccccc2)cc(OCc2ccc(CN3CCC[C@@H]3CO)cc2)c1                           56.0
CN(C)CCOc1ccc(cc1)C(=C(/CCCl)c1ccccc1)\c1ccccc1                                      38.0
CC1(C)Oc2ccc3C4=C[C@@]56NC(=O)[C@]7(CCCN7C5=O)C[C@H]6C(C)(C)C4=[N+]([O-])c3c2C=C1    66.0
C[C@@H]1CC(=O)NN=C1c1ccc(N)c(c1)[N+]([O-])=O                                       

In [16]:
# Boolean indicator: at least one bit set?
has_any_bit = row_bit_sum > 0            # Series[bool], index matches df_fp

# ── quick summaries ────────────────────────────────────────────────────
print("Total SMILES:", len(has_any_bit))
print("Rows with at least one bit set:", has_any_bit.sum())
print("Rows with all-zero fingerprints :", (~has_any_bit).sum())

# ── inspect the all-zero cases (if any) ────────────────────────────────
zeros_df = df_fp.loc[~has_any_bit]
print("\nAll-zero fingerprints:")
display(zeros_df.head())                 # show a few; adjust/head() as needed

Total SMILES: 17766
Rows with at least one bit set: 17766
Rows with all-zero fingerprints : 0

All-zero fingerprints:


,bit_0,bit_1,bit_2,bit_3,bit_4,bit_5,bit_6,bit_7,bit_8,bit_9,bit_10,bit_11,bit_12,bit_13,bit_14,bit_15,bit_16,bit_17,bit_18,bit_19,bit_20,bit_21,bit_22,bit_23,bit_24,...,bit_2023,bit_2024,bit_2025,bit_2026,bit_2027,bit_2028,bit_2029,bit_2030,bit_2031,bit_2032,bit_2033,bit_2034,bit_2035,bit_2036,bit_2037,bit_2038,bit_2039,bit_2040,bit_2041,bit_2042,bit_2043,bit_2044,bit_2045,bit_2046,bit_2047
canonical_smiles,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
